# NB12 — P4 Proper AusMicrobiome + NGSA Replication (140-KO Gene List)

**Purpose:** Pre-specified replication of the original AusMicrobiome + NGSA analysis  
(microbeatlas project, Cu β = −0.010 p = 0.019; Zn β = −0.010 p = 0.027; Pb β = −0.009  
p = 0.040; Ni β = −0.009 p = 0.046) using the comprehensive Tier 1+2 gene list (140 KOs).  
Response updated from `biome_H_std` to `mean_levins_B_std` (from genus_trait_table.csv).

**Hypothesis:** β < 0 for each metal (higher metal concentration → narrower niche breadth).  
This is identical to the original analysis; only the gene list changes (94 KOs → 140 KOs).

**Model:** PGLS `mean_levins_B_std ~ ngsa_Metal_ppm_z` × 5 metals (Cu, Zn, Pb primary; Ni, Co secondary).  
Pagel's λ estimated by ML. BH-FDR across 5 metals.

**Pre-registration:** All decisions locked before running. Run once; no iterative tuning.

## Block 0 — Imports

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Project paths
PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
MICROBEATLAS = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology')
DATA = PROJECT / 'data'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(PROJECT / 'scripts'))
from pgls_utils import run_pgls, pgls_results_table, fdr_correct

MIN_N = 50  # pre-specified minimum genus count
PRIMARY_METALS = ['Cu', 'Zn', 'Pb']   # primary
SECONDARY_METALS = ['Ni', 'Co']        # secondary
ALL_METALS = PRIMARY_METALS + SECONDARY_METALS

# Pre-specified original results (microbeatlas project, biome_H_std, n=482)
ORIGINAL = {
    'Cu': {'beta': -0.010, 'p': 0.019},
    'Zn': {'beta': -0.010, 'p': 0.027},
    'Pb': {'beta': -0.009, 'p': 0.040},
    'Ni': {'beta': -0.009, 'p': 0.046},
    'Co': {'beta': +0.002, 'p': 0.658},
}

print('Imports OK')
print(f'Project: {PROJECT}')
print(f'GTDB tree: {TREE_BAC.exists()}')

Imports OK
Project: /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology
GTDB tree: True


## Block 1 — Load Data

In [2]:
# Comprehensive project: 482 AusMicrobiome genera × 140-KO density + mean_levins_B_std
comp_df = pd.read_csv(DATA / '02_ngsa_pgls_input.csv')
print(f'Comprehensive NB02 input: {comp_df.shape} rows × cols')
print(comp_df.columns.tolist())
print(comp_df.head(2))

Comprehensive NB02 input: (482, 7) rows × cols
['genus_lower', 'mean_levins_B_std', 'predictor_z', 'ko_per_mb_tier1_z', 'ko_per_mb_tier2_z', 'phylum', 'kingdom']
     genus_lower  mean_levins_B_std  predictor_z  ko_per_mb_tier1_z  \
0    abiotrophia           0.046450    -0.535036          -0.005288   
1  acaryochloris           0.193421    -1.049217          -0.546585   

   ko_per_mb_tier2_z         phylum   kingdom  
0          -0.183227     Firmicutes  Bacteria  
1          -1.021718  Cyanobacteria  Bacteria  


In [3]:
# Microbeatlas project: 482 genera × NGSA metal concentrations (AusMicrobiome spatial join)
micro_df = pd.read_csv(
    MICROBEATLAS / 'data/aus_microbiome/aus_levinsB_ngsa_pgls_input.csv'
)
print(f'Microbeatlas AusMicrobiome+NGSA: {micro_df.shape} rows × cols')
print(micro_df.columns.tolist())
print(micro_df[['genus_lower', 'ngsa_Cu_ppm', 'ngsa_Zn_ppm', 'ngsa_Pb_ppm',
                 'ngsa_Ni_ppm', 'ngsa_Co_ppm']].head(2))

Microbeatlas AusMicrobiome+NGSA: (482, 15) rows × cols
['genus_lower', 'biome_H_std', 'ko_per_mb_total_z', 'ko_per_mb_tier1_z', 'ko_per_mb_tier2_z', 'ngsa_Cu_ppm', 'ngsa_Zn_ppm', 'ngsa_Pb_ppm', 'ngsa_Ni_ppm', 'ngsa_Co_ppm', 'ngsa_Cu_ppm_z', 'ngsa_Zn_ppm_z', 'ngsa_Pb_ppm_z', 'ngsa_Ni_ppm_z', 'ngsa_Co_ppm_z']
     genus_lower  ngsa_Cu_ppm  ngsa_Zn_ppm  ngsa_Pb_ppm  ngsa_Ni_ppm  \
0    abiotrophia     13.35000    46.550000       14.670    15.750000   
1  acaryochloris     22.43617    51.836735       19.205    26.985106   

   ngsa_Co_ppm  
0        7.100  
1       10.954  


## Block 2 — Build P4 PGLS Input

In [4]:
# Join on genus_lower
# comp_df has: mean_levins_B_std, predictor_z (140-KO density z), ko_per_mb_tier1_z, ko_per_mb_tier2_z
# micro_df has: ngsa_Cu/Zn/Pb/Ni/Co_ppm (raw concentrations, AusMicrobiome spatial join)

metal_cols = [f'ngsa_{m}_ppm' for m in ALL_METALS]
merged = comp_df.merge(
    micro_df[['genus_lower'] + metal_cols],
    on='genus_lower',
    how='inner'
)

print(f'Before join: comp={len(comp_df)}, micro={len(micro_df)}')
print(f'After inner join: {len(merged)} genera')
print()
print('Columns:', merged.columns.tolist())
print()
print('NA counts per metal:')
for m in ALL_METALS:
    col = f'ngsa_{m}_ppm'
    print(f'  {m}: {merged[col].isna().sum()} NAs')

Before join: comp=482, micro=482
After inner join: 482 genera

Columns: ['genus_lower', 'mean_levins_B_std', 'predictor_z', 'ko_per_mb_tier1_z', 'ko_per_mb_tier2_z', 'phylum', 'kingdom', 'ngsa_Cu_ppm', 'ngsa_Zn_ppm', 'ngsa_Pb_ppm', 'ngsa_Ni_ppm', 'ngsa_Co_ppm']

NA counts per metal:
  Cu: 0 NAs
  Zn: 0 NAs
  Pb: 0 NAs
  Ni: 0 NAs
  Co: 0 NAs


In [5]:
# Z-score metal concentrations on the joined dataset
# (re-standardise to this specific sample, consistent with how biome_H_std was used in original)
for m in ALL_METALS:
    col = f'ngsa_{m}_ppm'
    z_col = f'ngsa_{m}_ppm_z'
    valid = merged[col].dropna()
    merged[z_col] = (merged[col] - valid.mean()) / valid.std()

print('Z-scored metal columns added:')
z_cols = [f'ngsa_{m}_ppm_z' for m in ALL_METALS]
print(merged[z_cols].describe().round(3))

Z-scored metal columns added:
       ngsa_Cu_ppm_z  ngsa_Zn_ppm_z  ngsa_Pb_ppm_z  ngsa_Ni_ppm_z  \
count        482.000        482.000        482.000        482.000   
mean           0.000         -0.000         -0.000         -0.000   
std            1.000          1.000          1.000          1.000   
min           -1.743         -2.910         -1.092         -2.418   
25%           -0.434         -0.530         -0.460         -0.531   
50%           -0.214         -0.120         -0.241         -0.196   
75%            0.209          0.416          0.057          0.318   
max           10.554          7.672         12.102          6.649   

       ngsa_Co_ppm_z  
count        482.000  
mean          -0.000  
std            1.000  
min           -3.045  
25%           -0.460  
50%           -0.153  
75%            0.439  
max            4.889  


In [6]:
# Confirm response variable
print('Response: mean_levins_B_std')
print(merged['mean_levins_B_std'].describe().round(4))
print(f'NAs: {merged["mean_levins_B_std"].isna().sum()}')

Response: mean_levins_B_std
count    482.0000
mean       0.1287
std        0.0964
min        0.0001
25%        0.0432
50%        0.1216
75%        0.1950
max        0.3939
Name: mean_levins_B_std, dtype: float64
NAs: 0


## Block 3 — Run PGLS for Each Metal

In [7]:
# Pre-specified PGLS model: mean_levins_B_std ~ ngsa_Metal_ppm_z
# Pagel's lambda estimated by ML on GTDB r214 bacterial genus tree
# Run once — no tuning

results = []

for metal in ALL_METALS:
    pred_col = f'ngsa_{metal}_ppm_z'
    sub = merged[['genus_lower', 'mean_levins_B_std', pred_col]].dropna()
    print(f'\n--- {metal} (n={len(sub)}) ---')
    
    if len(sub) < MIN_N:
        print(f'SKIP: n={len(sub)} < MIN_N={MIN_N}')
        results.append({
            'metal': metal,
            'n_genera': len(sub),
            'lambda_est': np.nan,
            'beta': np.nan,
            'SE': np.nan,
            'p_raw': np.nan,
            'error': f'n < MIN_N ({len(sub)} < {MIN_N})'
        })
        continue
    
    try:
        res = run_pgls(
            sub, TREE_BAC,
            response='mean_levins_B_std',
            predictors=[pred_col],
            taxon_col='genus_lower',
            label=f'P4_{metal}',
            min_n=MIN_N
        )
        print(f'  λ = {res["lambda_est"]:.4f}, β = {res["beta"]:+.5f}, SE = {res["SE"]:.5f}, p = {res["p_value"]:.4g}')
        results.append({
            'metal': metal,
            'n_genera': res['n'],
            'lambda_est': res['lambda_est'],
            'beta': res['beta'],
            'SE': res['SE'],
            'p_raw': res['p_value'],
            'error': None
        })
    except Exception as exc:
        print(f'  ERROR: {exc}')
        results.append({
            'metal': metal,
            'n_genera': len(sub),
            'lambda_est': np.nan,
            'beta': np.nan,
            'SE': np.nan,
            'p_raw': np.nan,
            'error': str(exc)
        })

res_df = pd.DataFrame(results)
print('\nRaw results:')
print(res_df.to_string(index=False))


--- Cu (n=482) ---


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  λ = 0.3186, β = -0.01060, SE = 0.00438, p = 0.01595

--- Zn (n=482) ---


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  λ = 0.3182, β = -0.01064, SE = 0.00442, p = 0.01637

--- Pb (n=482) ---


...

  λ = 0.3264, β = -0.00929, SE = 0.00437, p = 0.034

--- Ni (n=482) ---


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  λ = 0.3541, β = -0.00872, SE = 0.00442, p = 0.04905

--- Co (n=482) ---


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  λ = 0.3532, β = +0.00126, SE = 0.00442, p = 0.7758

Raw results:
metal  n_genera  lambda_est      beta       SE    p_raw error
   Cu       482      0.3186 -0.010604 0.004384 0.015950  None
   Zn       482      0.3182 -0.010638 0.004416 0.016368  None
   Pb       482      0.3264 -0.009291 0.004370 0.034002  None
   Ni       482      0.3541 -0.008721 0.004420 0.049046  None
   Co       482      0.3532  0.001258 0.004416 0.775847  None


## Block 4 — BH-FDR and Comparison Table

In [ ]:
# BH FDR across all 5 metals
valid_mask = res_df['p_raw'].notna()
if valid_mask.sum() > 0:
    _, q_bh, _, _ = multipletests(res_df.loc[valid_mask, 'p_raw'].values, method='fdr_bh')
    res_df.loc[valid_mask, 'q_bh'] = q_bh
else:
    res_df['q_bh'] = np.nan

# Fix: pre-specified direction is β < 0 for ALL metals (including Co).
# Co was expected to be null in the original 94-KO analysis, but the pre-specified
# direction for this analysis is β < 0. β > 0 is directionally inconsistent.
res_df['direction_consistent'] = res_df['beta'] < 0

# FDR-corrected significance (q_BH ≤ 0.05)
res_df['significant_fdr05'] = res_df['q_bh'].apply(lambda q: q <= 0.05 if pd.notna(q) else False)

print('Final P4 results (BH-corrected):')
print(res_df[['metal', 'n_genera', 'lambda_est', 'beta', 'SE', 'p_raw', 'q_bh', 'direction_consistent', 'significant_fdr05']]
      .to_string(index=False, float_format=lambda x: f'{x:.5f}'))

In [9]:
# Comparison table vs original
print('=' * 90)
print('COMPARISON: Original (microbeatlas, biome_H_std, 94-KO) vs P4 (comprehensive, mean_levins_B_std, 140-KO)')
print('=' * 90)
print(f'{"Metal":<6} {"Orig β":>10} {"Orig p":>10} {"P4 β":>10} {"P4 p_raw":>10} {"P4 q_BH":>10} {"Dir OK":>8}')
print('-' * 90)

consistent_count = 0
for _, row in res_df.iterrows():
    m = row['metal']
    orig = ORIGINAL[m]
    ok = str(row['direction_consistent'])
    if row['direction_consistent']:
        consistent_count += 1
    beta_str = f"{row['beta']:+.5f}" if pd.notna(row['beta']) else 'NA'
    p_str = f"{row['p_raw']:.4g}" if pd.notna(row['p_raw']) else 'NA'
    q_str = f"{row['q_bh']:.4g}" if pd.notna(row.get('q_bh')) else 'NA'
    print(f"{m:<6} {orig['beta']:>+10.3f} {orig['p']:>10.4g} {beta_str:>10} {p_str:>10} {q_str:>10} {ok:>8}")

print('-' * 90)
print(f'Direction consistent: {consistent_count}/{len(res_df.dropna(subset=["beta"]))}')

COMPARISON: Original (microbeatlas, biome_H_std, 94-KO) vs P4 (comprehensive, mean_levins_B_std, 140-KO)
Metal      Orig β     Orig p       P4 β   P4 p_raw    P4 q_BH   Dir OK
------------------------------------------------------------------------------------------
Cu         -0.010      0.019   -0.01060    0.01595    0.04092     True
Zn         -0.010      0.027   -0.01064    0.01637    0.04092     True
Pb         -0.009       0.04   -0.00929      0.034    0.05667     True
Ni         -0.009      0.046   -0.00872    0.04905    0.06131     True
Co         +0.002      0.658   +0.00126     0.7758     0.7758     True
------------------------------------------------------------------------------------------
Direction consistent: 5/5


## Block 5 — Save Results

In [10]:
# Save to pre-specified output file
out_path = DATA / 'ngsa_replication_proper_comprehensive.csv'
save_cols = ['metal', 'n_genera', 'lambda_est', 'beta', 'SE', 'p_raw', 'q_bh', 'direction_consistent']
res_df[save_cols].to_csv(out_path, index=False)
print(f'Saved {len(res_df)} rows to {out_path}')

Saved 5 rows to /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/data/ngsa_replication_proper_comprehensive.csv


## Block 6 — Interpretation and INTERPRETATION_TABLE.md Update

In [ ]:
# Print interpretation summary — use q_BH ≤ 0.05 for significance (pre-specified threshold)
n_sig_fdr = int(res_df['significant_fdr05'].sum())
n_dir = int(res_df['direction_consistent'].sum())
n_valid = int(res_df['beta'].notna().sum())

print('INTERPRETATION SUMMARY (P4)')
print('=' * 60)
print(f'Genera analysed: {res_df["n_genera"].iloc[0] if n_valid > 0 else "N/A"}')
print(f'Metals tested: {n_valid}/5')
print(f'Direction consistent (β < 0): {n_dir}/{n_valid}')
print(f'FDR-significant (q_BH ≤ 0.05): {n_sig_fdr}/{n_valid}')
print()
print('Pre-specified expectation: β < 0 for Cu, Zn, Pb, Ni; Co null')
print()

# Classification uses FDR-corrected significance (q_BH ≤ 0.05), not p_raw
if n_sig_fdr >= 3 and n_dir >= 4:
    classification = 'REPLICATES original — direction and FDR-significance consistent'
elif n_dir >= 4 and n_sig_fdr >= 1:
    classification = 'PARTIAL REPLICATION — direction consistent, Cu and Zn significant at FDR 5%'
elif n_dir >= 3:
    classification = 'DIRECTIONALLY CONSISTENT — FDR significance lost'
else:
    classification = 'FAILS TO REPLICATE — direction inconsistent'

print(f'Classification: {classification}')

In [ ]:
# Update INTERPRETATION_TABLE.md — replace P4 section with corrected version
interp_path = PROJECT / 'INTERPRETATION_TABLE.md'
with open(interp_path, 'r') as f:
    content = f.read()

# Build result rows for the table — use q_BH ≤ 0.05 for significance flag
table_rows = []
for _, row in res_df.iterrows():
    m = row['metal']
    if pd.notna(row['beta']):
        b = f"{row['beta']:+.5f}"
        se = f"{row['SE']:.5f}"
        p = f"{row['p_raw']:.4g}"
        q = f"{row['q_bh']:.4g}" if pd.notna(row.get('q_bh')) else 'NA'
        n = int(row['n_genera'])
        lam = f"{row['lambda_est']:.4f}" if pd.notna(row['lambda_est']) else 'NA'
        ok = 'Yes' if row['direction_consistent'] else 'No'
        sig = 'SIGNIFICANT' if row['significant_fdr05'] else 'NON-SIGNIFICANT'
    else:
        b = se = p = q = lam = ok = 'NA'
        n = int(row['n_genera'])
        sig = f'FAILED: {row.get("error", "")}'
    table_rows.append(f'| {m} | {n} | {lam} | {b} | {se} | {p} | {q} | {ok} | {sig} |')

p4_section = f'''
---

## P4 — Proper AusMicrobiome+NGSA Replication (Notebook 12)

**Analysis:** PGLS `mean_levins_B_std ~ ngsa_Metal_ppm_z` for each of 5 metals.  
**Gene list:** Tier 1+2 (140 KOs) — same genera as P3 but metal-concentration predictor (not KO density).  
**Dataset:** AusMicrobiome OTU genera × NGSA ICP-MS soil concentrations (AusMicrobiome spatial join, ≤200 km).  
**Pre-specified direction:** β < 0 for all metals (higher metal → narrower niche).  
**Original reference:** Cu β = −0.010 p = 0.019; Zn β = −0.010 p = 0.027; Pb β = −0.009 p = 0.040; Ni β = −0.009 p = 0.046.

**Note on λ:** λ = 0.32–0.35 across all metals (vs. λ = 0.757 in P1). The lower λ likely reflects the reduced phylogenetic diversity of a continentally restricted genus set and the different predictor (environmental metal concentration vs. genomic density).

**Note on independence:** Cu, Zn, Pb, Ni concentrations are spatially co-deposited in the NGSA soil geochemistry data; these five tests are not fully independent. Effective number of independent tests ≈ 1–2. Significance statements should be interpreted accordingly.

| Metal | n | λ | β | SE | p_raw | q_BH | Dir consistent | Result |
|-------|---|---|---|----|-------|------|----------------|--------|
{chr(10).join(table_rows)}

**Classification:** {classification}
'''

# Replace existing P4 section or append
import re
p4_pattern = re.compile(r'---\s*\n## P4 — Proper AusMicrobiome.*?(?=\n---|\Z)', re.DOTALL)
if p4_pattern.search(content):
    new_content = p4_pattern.sub(p4_section.strip(), content)
    with open(interp_path, 'w') as f:
        f.write(new_content)
    print(f'Replaced P4 section in {interp_path}')
else:
    with open(interp_path, 'a') as f:
        f.write(p4_section)
    print(f'Appended P4 section to {interp_path}')